# Lab 3: Securely connect tools to your Agent with AgentCore Gateway 

## Overview

In this Lab, you will learn how to integrate tools available in your organization with the Product Launch Agent using the Amazon Bedrock Gateway.

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) is an open protocol that standardizes how applications provide tools and context to Large Language Models (LLMs).

With [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html), developers can convert APIs, Lambda functions, and existing services into MCP-compatible tools and make them available to agents through Gateway endpoints with just a few lines of code.


**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional product launch agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Current):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4:** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5:** Build User Interface - Create a product manager-facing application


### Why AgentCore Gateway & Tool Sharing Matter

Current State (Lab 1-2): Each agent has its own copy of tools. I practice that is not scalable and leads to:

- Code duplication across different agents
- Inconsistent tool behavior and maintenance overhead
- No centralized security or access control
- Difficulty scaling to multiple use cases

After this lab, we will have centralized, reusable tools that can serve:

- **Product Launch Agent** (our current use case) - needs market data and web search
- **Risk Assessment Agent** - needs same data sources for analysis
- **Sales Agent** - needs product information and customer data
- **Compliance Agent** - needs regulatory checking capabilities

and other use cases.

### 💡 Real-World Extension: External MCP Servers

While this lab uses Lambda functions to demonstrate Gateway, **AgentCore Gateway can also connect to external MCP servers**:

**Financial Services MCP Servers:**
```json
{
  "mcpServers": {
    "mastercard-open-banking": {
      "command": "npx",
      "args": [
        "-y",
        "@mastercard/developers-mcp",
        "--service=https://developer.mastercard.com/open-banking-us/documentation/"
      ]
    }
  }
}
```

**Benefits for Product Launch:**
- Access real-time banking data for market analysis
- Connect to compliance databases for regulatory checks
- Integrate with financial data providers
- Use industry-standard MCP servers without custom code

The Gateway pattern you learn here applies to any MCP server! 

### Adding secure authentication with AgentCore Identity

Additionally, AgentCore Gateway requires you to securely authenticate both inbound and outbound connections. [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html) provides seamless agent identity and access management across AWS services and third-party applications such as Slack and Zoom while supporting any standard identity providers such as Okta, Entra, and Amazon Cognito. In this lab we will see how AgentCore Gateway integrates with AgentCore Identity to provide secure connections via inbound and outbound authentication. 

For the inbound authentication, the AgentCore Gateway analyzes the OAuth token passed during invocation to decide allow or deny the access to a tool in the gateway. If a tool needs access to external resources, the AgentCore Gateway can use outbound authentication via API Key, IAM or OAuth Token to allow or deny the access to the external resource.

During the inbound authorization flow, an agent or the MCP client calls an MCP tool in the AgentCore Gateway adding an OAuth access token (generated from the user’s IdP). AgentCore Gateway then validates the OAuth access token and performs inbound authorization.

If the tool running in AgentCore Gateway needs to access external resources, OAuth will retrieve credentials of downstream resources using the resource credential provider for the Gateway target. AgentCore Gateway pass the authorization credentials to the caller to get access to the downstream API.


## Architecture for Lab 3


*Market research tool is now centralized in AgentCore Gateway with secure identity-based access control. Multiple agents and use cases can share the same tool securely. We will also reuse the `check_compliance_status()` and `get_pm_profile()` tools built for other financial applications. `market_research()` and `create_marketing_poster()` remain as local tools as they are specific to the product launch use case* 

### Key Features
- **Seamlessly integrate AWS Lambda functions:** This example shows how to integrate your Agent with existing AWS Lambda functions to check the warranty of an item and to get the product manager profile using Amazon Bedrock AgentCore Gateway.
- **Secure your Gateway endpoint with Inbound Auth**: Only an Agent providing a valid JWT token can connect to the endpoint to use the tools
- **Configure the Agent to use the MCP endpoint**: The Agent gets a valid JWT token and uses it to connect to the MCP endpoint provided by AgentCore Gateway

## Prerequisites

* Python 3.12+
* AWS credentials configured
* Anthropic Claude 3.7 enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* Complete Lab 2 Add memory to the Product Launch Agent
* These resources are created for you within an AWS workshop account
    - AWS Lambda function 
    - AWS Lambda Execution IAM Role
    - AgentCore Gateway IAM Role
    - DynamoDB tables used by the AWS Lambda function
    - Cognito User Pool and User Pool Client
    
#### Not using an AWS workshop account? 

**Note:** If you are running this as a self-paced lab, you must create the CloudFormation resources.

**For AWS Workshop Participants:**
- Infrastructure is pre-configured ✅
- Skip the cell below and continue to Step 1

**For Self-Paced Learners:**

The `scripts/prereq.sh` script requires:
- `prerequisite/lambda/python/` - Lambda function code
- `prerequisite/infrastructure.yaml` - CloudFormation template
- `prerequisite/cognito.yaml` - Cognito setup template

These files are provided in AWS workshop environments. If you don't have them:
1. You can follow along conceptually (the notebook explains each step)
2. Or create your own Lambda functions based on the tool schemas shown in Step 4

If you have the prerequisite files, uncomment and run the cell below:

In [ ]:
!bash scripts/prereq.sh

## Step 1: Install and import required libraries

In [ ]:
# Install required packages
#%pip install strands-agents "boto3>=1.39.15" strands-agents-tools bedrock_agentcore ddgs -q
# Install required packages
%pip install -U -r requirements.txt -q

In [ ]:
# Import libraries
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
import os
import sys
import boto3
import json
from bedrock_agentcore.identity.auth import requires_access_token
from mcp.client.streamable_http import streamablehttp_client
import requests

from scripts.utils import get_ssm_parameter, put_ssm_parameter, load_api_spec, get_cognito_client_secret

sts_client = boto3.client('sts')

# Get AWS account details
REGION = boto3.session.Session().region_name

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("✅ Libraries imported successfully!")

## Step 2: Give our agent tools to access enterprise and external data
AgentCore Gateway simplifies agent tool integration in three key ways:

**Universal MCP Support:** Instantly make your tools compatible with any agent framework by exposing them through AgentCore Gateway's MCP standard

**Simple REST Integration:** Transform existing REST services into agent tools by just adding them as AgentCore Gateway targets

**Lambda Flexibility:** Expose Lambda functions as MCP endpoints that can call any API

### Three Types of Gateway Tools for Product Launch:

**1. Regulatory Compliance Tool (Lambda → Enterprise Database)**
```python
def check_regulatory_compliance(product_type: str, region: str):
    """Query enterprise compliance database for regulations by region"""
    # Returns: Federal regulations, state requirements, licenses, timeline
    # Example: TILA, FCRA, ECOA for auto loans in California
```

**2. Product Performance Tool (Lambda → Internal Analytics)**
```python
def get_product_performance_data(product_id: str, metrics: list):
    """Get existing product data from internal systems"""
    # Returns: Margins, market share, customer satisfaction, strategic vision
    # Example: AUTO_LOAN_2023 has 15% margin, 8% market share
```

**3. Banking Data Tool (Gateway → Mastercard MCP Server)**
```python
def get_banking_data(data_type: str, customer_id: str):
    """Connect to Mastercard Open Banking API via MCP"""
    # Returns: Real-time account data, transactions, spending patterns
    # Example: Customer spending on auto purchases, loan payment history
```

AgentCore Gateway populates the Lambda context with the tool name:
```python
extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
resource = extended_tool_name.split("___")[1]
```

**Why These Tools Matter for Product Launch:**
- **Compliance:** Avoid regulatory issues by checking requirements early
- **Performance Data:** Learn from existing products to improve new launches
- **Banking Data:** Understand customer behavior for better targeting

## Step 3: Centralize enterprise tools via Gateway
Now that we are developing an MCP server using AgentCore Gateway, we can centralize tools that multiple agents need. For financial product launch, we're centralizing three critical capabilities:

### Tool 1: Regulatory Compliance (Lambda → Compliance Database)
[Compliance Lambda](./prerequisite/lambda/python/compliance_check.py)
```python
def check_regulatory_compliance(product_type: str, region: str) -> dict:
    """Check regulatory requirements from enterprise compliance database.
    
    Returns federal and state regulations, required licenses, and timeline.
    Example: For auto_loan in US-CA, returns TILA, FCRA, ECOA, CA state reqs.
    """
    # Query enterprise compliance database
    regulations = compliance_db.query(product_type, region)
    return {
        "federal_regulations": regulations.federal,
        "state_requirements": regulations.state,
        "required_licenses": regulations.licenses,
        "compliance_timeline": "4-6 weeks"
    }
```

### Tool 2: Product Performance Data (Lambda → Analytics Database)
[Product Data Lambda](./prerequisite/lambda/python/product_data.py)
```python
def get_product_performance_data(product_id: str, metrics: list) -> dict:
    """Get existing product performance from internal analytics.
    
    Returns margins, market share, customer satisfaction, strategic vision.
    Example: AUTO_LOAN_2023 → 15% margin, 8% market share, 4.2/5 satisfaction.
    """
    # Query internal product analytics database
    data = analytics_db.get_product(product_id)
    return {
        "margin": data.profit_margin,
        "market_share": data.market_position,
        "customer_satisfaction": data.csat_score,
        "strategic_vision": data.product_vision
    }
```

### Tool 3: Banking Data (Gateway → Mastercard MCP Server)
[External MCP Integration](./prerequisite/lambda/python/banking_mcp.py)
```python
def get_banking_data(data_type: str, customer_id: str) -> dict:
    """Connect to Mastercard Open Banking API via MCP.
    
    Returns real-time account data, transactions, spending patterns.
    Used for market analysis and customer segmentation.
    """
    # Connect to external MCP server
    mcp_client = connect_to_mastercard_mcp()
    return mcp_client.get_data(data_type, customer_id)
```

**Why Gateway for These Tools?**
- **Compliance:** Risk, Legal, and Product teams all need regulatory data
- **Product Data:** Product, Sales, and Strategy teams need performance metrics
- **Banking Data:** Multiple teams need customer insights for targeting
- **Centralized:** One place to manage access, updates, and security

## Step 4: Create your function definition metadata
Lastly, we need to write tool schema which describes the tools implemented by your Lambda function.

This file has been already defined in [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json)

```json
[
    {
        "name": "check_regulatory_compliance",
        "description": "Check regulatory compliance requirements for financial products by country/region. Returns federal and state regulations, required licenses, and compliance timeline.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_type": {
                    "type": "string",
                    "description": "Type of financial product (e.g., auto_loan, personal_loan, credit_card)"
                },
                "region": {
                    "type": "string",
                    "description": "Country or state code (e.g., US, US-CA, UK)"
                }
            },
            "required": [
                "product_type",
                "region"
            ]
        }
    },
    {
        "name": "get_product_performance_data",
        "description": "Get existing product performance data including margins, market share, customer satisfaction, and strategic vision from enterprise database.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_id": {
                    "type": "string",
                    "description": "Product identifier (e.g., AUTO_LOAN_2023, PERSONAL_LOAN_PRIME)"
                },
                "metrics": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["margin", "market_share", "customer_satisfaction", "revenue", "vision"]
                    },
                    "description": "Specific metrics to retrieve"
                }
            },
            "required": [
                "product_id"
            ]
        }
    },
    {
        "name": "get_banking_data",
        "description": "Connect to Mastercard Open Banking API to get real-time account data, transaction history, and customer financial profiles for market analysis.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "data_type": {
                    "type": "string",
                    "enum": ["account_info", "transactions", "customer_profile", "spending_patterns"],
                    "description": "Type of banking data to retrieve"
                },
                "customer_id": {
                    "type": "string",
                    "description": "Customer identifier for data retrieval"
                }
            },
            "required": [
                "data_type"
            ]
        }
    }
]
```

**Financial Product Launch Tools:**
1. **check_regulatory_compliance** - Enterprise compliance database for federal/state regulations
2. **get_product_performance_data** - Internal product analytics (margins, market share, vision)
3. **get_banking_data** - External MCP server (Mastercard Open Banking) for real-time market data

**Note:** The actual Lambda implementation uses generic parameter names, but the schema describes them for financial product launch. In production, you'd update the Lambda code to match these schemas.

## Step 5. Create your AgentCore Gateway

Now let's create the AgentCore Gateway to expose the Lambda function as MCP-compatible endpoint.

To validate the callers authorized to invoke our tools we need to configure the Inbound Auth.

Inbound Auth works using OAuth authorization, the standard for MCP servers. With OAuth the client application must authenticate with the OAuth authorizer before using the Gateway. Your client would receive an access token which is used at runtime.

You need to specify an OAuth discovery server and client IDs. The Cloudformation provided with the workshop already provisioned the Cognito UserPool and UserPoolClient and it stored the discovery URL and the Client ID in dedicated SSM parameters.

In [ ]:
gateway_name = "productlaunch-gw"

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            get_ssm_parameter("/app/productlaunch/agentcore/machine_client_id")
        ],
        "discoveryUrl": get_ssm_parameter("/app/productlaunch/agentcore/cognito_discovery_url")
    }
}

try:
    # create new gateway
    print(f"Creating gateway in region {REGION} with name: {gateway_name}")

    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn= get_ssm_parameter("/app/productlaunch/agentcore/gateway_iam_role"),
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=auth_config,
        description="Product Launch AgentCore Gateway",
    )

    gateway_id = create_response["gatewayId"]

    gateway = {
        "id": gateway_id,
        "name": gateway_name,
        "gateway_url": create_response["gatewayUrl"],
        "gateway_arn": create_response["gatewayArn"],
    }
    put_ssm_parameter("/app/productlaunch/agentcore/gateway_id", gateway_id)

    print(f"✅ Gateway created successfully with ID: {gateway_id}")

except Exception as e:
    # If gateway exists, collect existing gateway ID from SSM
    existing_gateway_id = get_ssm_parameter("/app/productlaunch/agentcore/gateway_id")
    print(f"Found existing gateway with ID: {existing_gateway_id}")
    
    # Get existing gateway details
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=existing_gateway_id)
    gateway = {
        "id": existing_gateway_id,
        "name": gateway_response["name"],
        "gateway_url": gateway_response["gatewayUrl"],
        "gateway_arn": gateway_response["gatewayArn"],
    }
    gateway_id = gateway['id']

In [ ]:
# wait for the gateway to be active

import time
for i in range(30, 0, -1):
    print(f"⏳ Waiting... {i} seconds remaining", end='\r')
    time.sleep(1)
print("✅ Done waiting" + " " * 30)

## Step 6. Add the Lambda function Target
Now we will use the previously defined function definitions from [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json) to create a Lambda target within our Agent Gateway. This will define the tools that your gateway will host.

Gateway allows you to attach multiple targets to a Gateway and you can change the targets / tools attached to a gateway at any point. Each target can have its own credential provider, but Gateway becomes a single MCP URL enabling access to all of the relevant tools for an agent across myriad APIs.

In [ ]:
def load_api_spec(file_path: str) -> list:
    with open(file_path, "r") as f:
        data = json.load(f)
        
    if not isinstance(data, list):
        raise ValueError("Expected a list in the JSON file")
    return data

try:
    api_spec_file = "./prerequisite/lambda/api_spec.json"

    # Validate API spec file exists
    if not os.path.exists(api_spec_file):
        print(f"❌ API specification file not found: {api_spec_file}")
        sys.exit(1)

    api_spec = load_api_spec(api_spec_file)
 
    # Use Cognito for Inbound OAuth to our Gateway
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": get_ssm_parameter("/app/productlaunch/agentcore/lambda_arn"),
                "toolSchema": {"inlinePayload": api_spec},
            }
        }
    }


    # Create gateway target
    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaUsingSDK",
        description="Lambda Target using SDK",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=credential_config,
    )

    print(f"✅ Gateway target created: {create_target_response['targetId']}")

except Exception as e:
    print(f"❌ Error creating gateway target: {str(e)}")

## Add our new tools to our  agent
Here we integrate our authentication token from Cognito into an MCPClient from Strands SDK to create an MCP Server object to integrate with our Strands Agent

In [ ]:
def get_token(client_id: str, client_secret: str, scope_string: str, url: str) -> dict:
    try:
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        data = {
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope_string,

        }
        response = requests.post(url, headers=headers, data=data)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as err:
        return {"error": str(err)}

###  Set up a secure MCP client object

In [ ]:
gateway_access_token = get_token(
    get_ssm_parameter("/app/productlaunch/agentcore/machine_client_id"),
    get_cognito_client_secret(),
    get_ssm_parameter("/app/productlaunch/agentcore/cognito_auth_scope"),
    get_ssm_parameter("/app/productlaunch/agentcore/cognito_token_url"))

print(f"Gateway Endpoint - MCP URL: {gateway['gateway_url']}")

# Set up MCP client
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway['gateway_url'],
        headers={"Authorization": f"Bearer {gateway_access_token['access_token']}"},
    )
)

### Add External MCP Server (External Remote MCP Server) - Optional

In addition to the Gateway tools (Lambda functions), we'll also connect to an external MCP server that provides  financial data and fraud prevention tools.

This demonstrates how you can combine:
- **Gateway Lambda tools** (compliance, product performance)
- **External MCP servers** (External Remote MCP Server for example MasterCard, Visa etc)
- **Local tools** (market research, marketing materials)

All accessible through a single agent!

In [ ]:
# create_gateway_target_response = gateway_client.create_gateway_target(
#     name='mastercard-mcp-server-target',
#     gatewayIdentifier=gateway_id,
#     targetConfiguration={
#         'mcp': {
#             'mcpServer': {
#                 'endpoint': 'developer.mastercard.com/mcp/endpoint' 
#             }
#         }
#     },
#     credentialProviderConfigurations=[
#         {
#             'credentialProviderType': 'OAUTH',
#             'credentialProvider': {
#                 'oauthCredentialProvider': {
#                     'providerArn': cognito_provider_arn,  # Your OAuth provider ARN
#                     'scopes': [
#                         'mastercard.api.access'  # Replace with actual Mastercard scopes
#                     ]
#                 }
#             }
#         },
#     ]
# )
#print(create_gateway_target_response)

## Step 7 Access tools in our agent 
Now we will create our Strands Agent using the AgentCore Gateway we built along with the resources from previous labs. Our agent now uses a mix of local tools via our Strands Agent and MCP tools via AgentCore Gateway

In [ ]:
# Import tools from previous labs
from lab_helpers.unified_market_research import market_research
from lab_helpers.marketing_tools import create_marketing_poster
from lab_helpers.lab2_memory import ProductLaunchMemoryHooks, create_or_get_memory_resource 
import uuid
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=REGION)

memory_id = create_or_get_memory_resource()
SESSION_ID = str(uuid.uuid4())
PM_ACTOR_ID = "pm_sarah_001"
memory_hooks = ProductLaunchMemoryHooks(memory_id, memory_client, PM_ACTOR_ID, SESSION_ID)

# System prompt for financial product launch agent
SYSTEM_PROMPT = """You are an expert financial product launch assistant helping product managers launch new financial products.
Your role is to:
- Provide market research and competitive intelligence
- Check compliance and regulatory requirements
- Access product manager profiles and launch history
- Generate marketing materials
- Be professional, data-driven, and strategic

You have access to both local tools and enterprise tools via AgentCore Gateway.
Use Gateway tools for compliance checks and market research that other teams also need."""

# Initialize the Bedrock model
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(
    model_id=model_id,
    temperature=0.3,
    region_name=REGION
)

try:
    mcp_client.start()
except Exception as e:
    print(f"Error initializing MCP client: {str(e)}")

# Combine local tools with Gateway tools
tools = (
    [
        market_research,           # Local: Unified market research
        create_marketing_poster,   # Local: Marketing material generation
    ]
    + mcp_client.list_tools_sync()  # Gateway: Compliance check, web search
)

# Create the financial product launch agent
agent = Agent(
    model=model,
    tools=tools,
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Financial Product Launch Agent with Gateway integration created successfully!")

## Step 8: Test the agent with MCP tool access to existing APIs”

Let's test our agent with sample queries to ensure all features work correctly.

In [ ]:
test_prompts = [
    # Test 1: List all available tools
    "List all of your tools",
    
    # Test 2: Check regulatory compliance (Gateway Lambda tool)
    "I'm launching an auto loan product in California. What are the regulatory compliance requirements?",
    
    # Test 3: Get existing product performance data (Gateway Lambda tool)
    "Get the performance data for our existing AUTO_LOAN_2023 product including margins and market share",
    
    
    # Test 4: Combined - Local + Gateway tools
    "Research competitor rates using market research, then check what compliance requirements we need for a personal loan in Texas",
    
    # Test 5: Memory + Gateway integration
    "Based on my previous launches and our AUTO_LOAN_2023 performance data, what should I focus on for my next product?"
]

# Function to test the agent
def test_agent_responses(agent, prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = agent(prompt)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("-" * 50)

# Run the tests
test_agent_responses(agent, test_prompts)

print("\\n✅ Basic testing completed!")


### Congratulations! 🎉


You have successfully completed Lab 3: Securely connect tools to your Agent with AgentCore Gateway

What You Accomplished:

##### Tool Centralization & Reusability:

- Migrated web search from local tool to centralized AgentCore Gateway
- Integrated existing enterprise Lambda functions (warranty check, product manager profile)
- Created a shared tool infrastructure that multiple agent types can access

##### Enterprise-Grade Security:

- Implemented JWT-based authentication with Cognito integration
- Configured secure inbound authorization for gateway access
- Established identity-based access control for tool usage

##### Scalable Architecture Foundation:

- Built reusable tools that serve multiple use cases (product launch, sales, returns processing)
- Eliminated code duplication across different agents
- Created centralized management for tool updates and maintenance

##### Current Limitations (We'll fix these next!):

- **Local Development Environment** - Still running on your laptop, not production-ready
- **Limited Observability** - No comprehensive monitoring of agent behavior and performance
- **Manual Scaling** - Cannot automatically handle increased load or multiple concurrent users

##### 🚀 Bonus: Extending with External MCP Servers

Want to connect to real financial data? You can extend this Gateway to include external MCP servers:

**Example: Mastercard Open Banking**
```python
# Add external MCP server as Gateway target
external_mcp_config = {
    "mcp": {
        "externalServer": {
            "serverUrl": "https://your-mcp-server.com",
            "authConfig": {"type": "API_KEY"}
        }
    }
}
```

**Other Financial MCP Servers to Explore:**
- Mastercard Open Banking API
- Plaid Financial Data
- Compliance databases
- Market data providers

The Gateway pattern you learned applies to any MCP-compatible service!

##### Next Up: Lab 4 - Deploying to Production with AgentCore Runtime

In Lab 4, you'll transform your prototype into a production-ready system with:

- AgentCore Runtime for scalable agent deployment
- Comprehensive observability with metrics, logging, and tracing
- Auto-scaling capabilities to handle real-world traffic

### Resources
- [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/)
- [Mastercard Developers MCP](https://developer.mastercard.com/)
- [Strands Agents Documentation](https://github.com/strands-agents/sdk-python)